In [55]:
import os,glob,random,time
import numpy as np
import pandas as pd
from PIL import Image
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score,precision_recall_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torchvision import models,transforms
from torchvision.models import ResNet50_Weights

In [ ]:
root_dir = "./datasets/MVTecAD"
img_size = 256
device  = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
categories = ["bottle","cable","capsule","carpet","grid","hazelnut","leather","metal_nut","pill","screw","tile","toothbrush","transistor","wood","zipper"]

local_k = 3
SIGMA = 4

# OVERVIEW of this MutiHead Model and changes made from previous models

1. Mask Transformation : inrerpolated with nearest instead of bilinear to maintain strict 0,1 values for normal,defective regions instead of fractional values being created during bilinear interpolation
2. Feature Extraction from ResNet layers 2,3 is followed by average pooling to smoothen out the features 
3. Added positional encoding to give the model a sense of position of the pixels it is seeing which might help detect anomalies due to incorrect positiongs of components, etc
4. Added Feature Normalisation to normalise each channel across all training examples
5. Gaussian Filter is added during evaluation to reduce impact of noisy pixels


- Head A : ae and ae_pos
1. Add noise to input to prevent model from lazily learning identity mapping
2. ae then just proceeds as normal 1x1 conv based autoencoder like in previous models
3. ae_pos involves positional encoding being concated into the inputs for both encoder and decoder


- Head B : padim
1. Purely statistical model, same as Mahalanobis technique used for OOD detection
2. Choose random d channels from all the output channels of autoencoder and find means and covariances for each position across all training examples
3. For test examples, find mahalanobis distance to check how different the test image is from ID cluster and classify



# Data Loading

In [57]:
class MVTecDataset(Dataset):
    def __init__(self,root_dir,category,is_train=True,transform=None,mask_transform=None):
        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        self.defect_types = []
        self.transform = transform
        self.mask_transform = mask_transform
        
        if is_train:
            paths = sorted(glob.glob(os.path.join(root_dir,category,"train","good","*.png")))
            self.image_paths.extend(paths)
            self.labels.extend([0]*len(paths))
            self.mask_paths.extend([None]*len(paths))
            self.defect_types.extend(["good"*len(paths)])
        
        else:
            test_dir = os.path.join(root_dir,category,"test")
            for defect in os.listdir(test_dir):
                if defect.startswith('.'):
                    continue
                paths = sorted(glob.glob(os.path.join(test_dir,defect,"*.png")))
                self.image_paths.extend(paths)
                self.defect_types.extend([defect]*len(paths))

                if defect == "good":
                    self.labels.extend([0]*len(paths))
                    self.mask_paths.extend([None]*len(paths))
                else:
                    self.labels.extend([1]*len(paths))
                    for p in paths:
                        self.mask_paths.append(os.path.join(root_dir,category,"ground_truth",defect,os.path.basename(p).replace(".png","_mask.png")))
    
    def __len__(self):
            return len(self.image_paths)
    
    def __getitem__(self,idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        mask_path = self.mask_paths[idx]
        if mask_path and os.path.exists(mask_path):
            mask = Image.open(mask_path).convert("L")
            mask = self.mask_transform(mask) if self.mask_transform else transforms.ToTensor()(mask)
        else:
            mask = torch.zeros((1,image.shape[1],image.shape[2]))
        return image,self.labels[idx],mask

**Transforms**

Changes : mask transformation is interpolated using Nearest instead of Bilinear

In [58]:
img_trfm = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225])
    ])

mask_trfm = transforms.Compose([
    transforms.Resize((img_size,img_size),interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor()
])

# Feature Extractor from ResNet Layers

Changes : Average Pooling of features extracted to smoothen out the features

In [ ]:
class ResNetFeatureExtractor(nn.Module):
    def __init__(self,k=3):
        super().__init__()
        resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.feats_l2 = nn.Sequential(resnet.conv1,resnet.bn1,resnet.relu,resnet.maxpool,resnet.layer1,resnet.layer2)
        self.feats_l3 = resnet.layer3
        
        self.k = k
        self.pad = k//2
        
        self.eval()
        for param in self.parameters():
            param.requires_grad = False
            
    def train(self,mode=True):      
        return super().train(False)
    
    @torch.no_grad()
    def forward(self,x):
        f2 = self.feats_l2(x)
        f3 = self.feats_l3(f2)
                
        if self.k > 1:
            f2,f3 = F.avg_pool2d(f2,self.k,stride=1,padding=self.pad,count_include_pad=False),F.avg_pool2d(f3,self.k,stride=1,padding=self.pad,count_include_pad=False)
        f3 = F.interpolate(f3,size=f2.shape[2:],mode="bilinear",align_corners=False)
        return torch.cat([f2,f3],dim=1)      
    
@torch.no_grad()
def extract_features(dataset,extractor,batch_size=16,device="mps"):
    loader = DataLoader(dataset,batch_size=batch_size,shuffle=False,num_workers=0)
    feats,labels,masks = [],[],[]
    for X,y,m in loader:
        feats.append(extractor(X.to(device)).half().cpu())
        labels.append(y)
        masks.append((m>0).to(torch.uint8))
    return torch.cat(feats),torch.cat(labels).numpy(),torch.cat(masks)


# Feature Normalisation and Positional Encoding

In [60]:
class FeatureNormaliser:
    def fit(self,feats):
        n,s,ss = 0,0,0
        for i in range(0,len(feats),16):
            x = feats[i:min(len(feats),i+16)]
            s += x.sum(dim=(0,2,3),dtype=torch.float64)
            ss += (x**2).sum(dim=(0,2,3),dtype=torch.float64)
            n += x.shape[0]*x.shape[2]*x.shape[3]
        mean = s/n
        std = (ss/n - mean**2).clamp_min(0).sqrt()
        
        self.mean = mean.float().view(1,-1,1,1) # Per channel mean
        self.std = std.float().clamp_min(1e-2).view(1,-1,1,1) # Per channel std
        return self

    def __call__(self, x):
        return (x - self.mean.to(x.device))/self.std.to(x.device)

In [61]:
def positional_encoding(H,W,n_freq=4):
    ys,xs = torch.linspace(0,1,H),torch.linspace(0,1,W)
    gy,gx = torch.meshgrid(ys,xs,indexing="ij")
    chans = []
    
    for k in range(n_freq):
        f = (2**k)*np.pi #pi,2pi,4pi,8pi,...
        for g in (gx,gy):
            chans+=[torch.sin(f*g),torch.cos(f*g)]
    return torch.stack(chans)            

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)

## Head A: Denoising 1x1 Convolutional AutoEncoder

1536 -> 512 -> 128 -> 512 -> 1536 shape (still per-location, no spatial mixing) with:
- noise added to the input while the target stays clean otherwise network could lazily just learn identity mapping
- positional channels into both the encoder and the decoder, giving them to the decoder lets it learn "what feature is expected here", which is what catches a missing or misplaced part
- 60 epochs with a cosine schedule on cached features

Anomaly score per location is the mean squared error over the standardised channels.

In [62]:
def conv1x1_bn_relu(i,o):
    return [nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(inplace=True)]

class ConvAutoEncoder(nn.Module):
    def __init__(self,feat=1536,pos=0,hidden=512,bottleneck=128):
        super().__init__()
        self.pos = pos
        self.encoder = nn.Sequential(
            *conv1x1_bn_relu(feat+pos,hidden),*conv1x1_bn_relu(hidden,bottleneck)
        )
        self.decoder = nn.Sequential(
            *conv1x1_bn_relu(bottleneck+pos,hidden),nn.Conv2d(in_channels=hidden,out_channels=feat,kernel_size=1,stride=1)
        )
        
    def forward(self,x,positional=None):
        if self.pos:
            p = positional.unsqueeze(0).expand(x.shape[0],-1,-1,-1)
            z = self.encoder(torch.cat([x,p],1))
            return self.decoder(torch.cat([z,p],1))
        return self.decoder(self.encoder(x))

In [63]:
class AEhead:
    def __init__(self,use_pos =False,epochs=60,lr=1e-3,noise_std=0.1,batch_size=32,seed=42,verbose=False):
        self.use_pos,self.epochs,self.lr,self.noise_std,self.batch_size,self.seed,self.verbose = use_pos,epochs,lr,noise_std,batch_size,seed,verbose
        self.name = "ae_pos" if use_pos else "ae"
        
    def fit(self,train_feats):
        set_seed(self.seed)
        N,C,H,W = train_feats.shape
        
        self.norm = FeatureNormaliser().fit(train_feats)
        self.pos = positional_encoding(H,W).to(device) if self.use_pos else None
        self.model = ConvAutoEncoder(C,0 if (self.use_pos == False) else self.pos.shape[0]).to(device)
        
        optimizer = torch.optim.Adam(self.model.parameters(),lr=self.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=self.epochs)
        
        loader = DataLoader(train_feats,batch_size=self.batch_size,shuffle=True,drop_last=len(train_feats)>self.batch_size)
        
        for epoch in range(self.epochs):
            self.model.train()
            total_loss = 0
            for X in loader:
                X = self.norm(X.to(device).float())
                X_in = X + self.noise_std*torch.randn_like(X)
                loss = F.mse_loss(self.model(X_in,self.pos),X)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
            scheduler.step()
            if self.verbose and (epoch+1)%10 == 0:
                print(f"[{self.name}] Epoch {epoch + 1}/{self.epochs} : Loss {total_loss/len(loader):.4f}")
        return self

    @torch.no_grad()
    def score_map(self,feats):
        self.model.eval()
        x = self.norm(feats.to(device).float())
        
        return ((x-self.model(x,self.pos))**2).mean(1,keepdim=True).cpu()

# Head B : Per position Gaussian 
**Idea similar to OOD/3_Mahalanobis**

Due to RAM limitations, choose any d channels instead of all to find mean and covariance for every single pixel location

At test time, using these find mahalanobis distance for each pixel of test image from the pixel's ID cluster and classify on basis of it

In [ ]:
class GaussianHead:
    name = "padim"
    def __init__(self,d=200,eps=0.01,seed=42):
        self.d,self.eps,self.seed = d,eps,seed
    
    def fit(self,train_feats):
        N,C,H,W = train_feats.shape # H W N C
        self.indexes = torch.randperm(C,generator=torch.Generator().manual_seed(self.seed))[:self.d]
        self.norm = FeatureNormaliser().fit(train_feats)
        
        m,sd = self.norm.mean[:,self.indexes],self.norm.std[:,self.indexes]
        X = (train_feats[:,self.indexes].float()-m)/sd
        X = X.permute(2,3,0,1).reshape(H*W,N,self.d)
        
        self.mean = X.mean(1)
        
        inv = torch.empty(H*W,self.d,self.d)
        I = torch.eye(self.d,dtype=torch.float64)
        
        for s in range(0,H*W,128):
            Xc = (X[s:s+128]-self.mean[s:s+128,None,:]).double()
            cov = torch.einsum("pnd,pne->pde",Xc,Xc)/max(N-1,1) + self.eps*I
            inv[s:s+128] = torch.linalg.inv(cov).float()
        self.inv,self.H,self.W = inv,H,W
        return self    
    
    @torch.no_grad()
    def score_map(self,feats):
        B = feats.shape[0]
        m,sd = self.norm.mean[:,self.indexes], self.norm.std[:,self.indexes]
        x = (feats[:,self.indexes].float()-m)/sd               
        x = x.permute(2,3,0,1).reshape(self.H*self.W,B,self.d) - self.mean[:,None,:]
        m = (torch.einsum("pbd,pde->pbe",x,self.inv)*x).sum(-1)    
        return m.clamp_min(0).sqrt().reshape(self.H,self.W,B).permute(2,0,1).unsqueeze(1)

# Evaluation

- Gaussian noise is added since it helps eliminate pixel noise which can raise false alarms in img_scores and can help consolidate real defects
- Checking performance across multiple metrics

In [65]:
def evaluate(head,test_feats,labels,masks,defects,sigma=None):
    sigma = SIGMA if sigma is None else sigma
    maps = []
    for i in range(0,len(test_feats),16):
        m = head.score_map(test_feats[i:i+16])
        m = F.interpolate(m,size=(img_size,img_size),mode="bilinear",align_corners=False)
        maps.append(m.squeeze(1).numpy())
    amap = np.concatenate(maps)                                             
    if sigma > 0:
        amap = np.stack([gaussian_filter(a,sigma=sigma) for a in amap])

    img_scores = amap.reshape(len(amap),-1).max(1)
    img_auroc = roc_auc_score(labels,img_scores)
    pxl_auroc = roc_auc_score(masks.numpy().reshape(-1),amap.reshape(-1))

    p,r,_ = precision_recall_curve(labels,img_scores)
    best_f1 = float((2*p*r/(p+r+1e-12)).max())

    defects = np.asarray(defects)
    per_defect = {}
    for d in sorted(set(defects)-{"good"}):
        sel = (defects == d) | (defects == "good")
        per_defect[d] = roc_auc_score(labels[sel],img_scores[sel])

    return dict(img_auroc=img_auroc,pxl_auroc=pxl_auroc,f1=best_f1,per_defect=per_defect),amap

# Running train and test on a category

In [ ]:
def normalize_map(amap):
    min_val,max_val = amap.min(),amap.max()
    return (amap-min_val)/(max_val-min_val+1e-8)

def evaluate(head, test_feats, labels, masks, defects, sigma=None):
    sigma = SIGMA if sigma is None else sigma
    maps = []
    for i in range(0, len(test_feats), 16):
        m = head.score_map(test_feats[i:i+16])
        m = F.interpolate(m, size=(img_size, img_size), mode="bilinear", align_corners=False)
        maps.append(m.squeeze(1).numpy())
    
    amap = np.concatenate(maps)                                             
    if sigma > 0:
        amap = np.stack([gaussian_filter(a, sigma=sigma) for a in amap])

    img_scores = amap.reshape(len(amap), -1).max(1)
    img_auroc = roc_auc_score(labels, img_scores)
    pxl_auroc = roc_auc_score(masks.numpy().reshape(-1), amap.reshape(-1))

    p, r, _ = precision_recall_curve(labels, img_scores)
    best_f1 = float((2 * p * r / (p + r + 1e-12)).max())

    defects = np.asarray(defects)
    per_defect = {}
    for d in sorted(set(defects) - {"good"}):
        sel = (defects == d) | (defects == "good")
        per_defect[d] = roc_auc_score(labels[sel], img_scores[sel])

    return dict(img_auroc=img_auroc, pxl_auroc=pxl_auroc, f1=best_f1, per_defect=per_defect), amap

def run_category(category, seeds=(0,), heads=("ae", "ae_pos", "padim"), verbose=True):
    train_ds = MVTecDataset(root_dir, category, True, img_trfm)
    test_ds = MVTecDataset(root_dir, category, False, img_trfm, mask_trfm)
    extractor = ResNetFeatureExtractor(k=local_k).to(device)

    t0 = time.time()
    train_feats, _, _ = extract_features(train_ds, extractor)
    test_feats, labels, masks = extract_features(test_ds, extractor)
    if verbose:
        print(f"[{category}] {len(train_ds)} train / {len(test_ds)} test, features {tuple(train_feats.shape[1:])}, "
              f"extracted in {time.time() - t0:.0f}s")

    make = {"ae":     lambda s: AEhead(use_pos=False, seed=s),
            "ae_pos": lambda s: AEhead(use_pos=True, seed=s),
            "padim":  lambda s: GaussianHead(seed=s)}
    
    rows, maps = [], {}
    
    for name in heads:
        for seed in seeds:
            head = make[name](seed).fit(train_feats)
            res, amap = evaluate(head, test_feats, labels, masks, test_ds.defect_types)
            rows.append(dict(category=category, head=name, seed=seed, img_auroc=res["img_auroc"],
                             pxl_auroc=res["pxl_auroc"], f1=res["f1"], per_defect=res["per_defect"]))
            if seed == seeds[0]:
                maps[name] = amap
            if verbose:
                print(f"  {name:9s} seed {seed} | img_AUROC: {res['img_auroc']:.4f} | pxl_AUROC: {res['pxl_auroc']:.4f} | F1: {res['f1']:.3f}")
    
    if len(maps) > 1:
        ensemble_amap = np.mean([normalize_map(m) for m in maps.values()], axis=0)
        
        img_scores = ensemble_amap.reshape(len(ensemble_amap), -1).max(1)
        img_auroc = roc_auc_score(labels, img_scores)
        pxl_auroc = roc_auc_score(masks.numpy().reshape(-1), ensemble_amap.reshape(-1))
        
        p, r, _ = precision_recall_curve(labels, img_scores)
        best_f1 = float((2 * p * r / (p + r + 1e-12)).max())
        
        defects_arr = np.asarray(test_ds.defect_types)
        per_defect = {}
        for d in sorted(set(defects_arr) - {"good"}):
            sel = (defects_arr == d) | (defects_arr == "good")
            per_defect[d] = roc_auc_score(labels[sel], img_scores[sel])
            
        rows.append(dict(category=category, head="ensemble", seed=seeds[0], img_auroc=img_auroc,
                         pxl_auroc=pxl_auroc, f1=best_f1, per_defect=per_defect))
        
        if verbose:
            print(f"  {'ensemble':9s} seed {seeds[0]} | img_AUROC: {img_auroc:.4f} | pxl_AUROC: {pxl_auroc:.4f} | F1: {best_f1:.3f}")

    return rows, maps, test_ds, labels, masks

# Running train and test across all categories

In [71]:
SEEDS = (0,)
all_rows = []

for cat in categories:
    print("\n" + "-"*70)
    print(f"Running Category: {cat.capitalize()}")
    print("-" * 70)
    
    rows, _, _, _, _ = run_category(cat, seeds=SEEDS)
    all_rows += rows

df = pd.DataFrame(all_rows)

print("\n" + "="*70)
print("FINAL SUMMARY (All Heads & Ensemble):")
# Use pivot to make a clean comparison table: Categories as rows, Heads as columns
pivot_df = df.pivot(index="category", columns="head", values=["img_auroc", "pxl_auroc"])
print(pivot_df.round(4).to_string())
print("="*70)

df.drop(columns="per_defect", errors="ignore").to_csv("results_CNNbased_MultiHead.csv", index=False)



----------------------------------------------------------------------
Running Category: Bottle
----------------------------------------------------------------------
[bottle] 209 train / 83 test, features (1536, 32, 32), extracted in 4s
  ae        seed 0 | img_AUROC: 1.0000 | pxl_AUROC: 0.9779 | F1: 1.000
  ae_pos    seed 0 | img_AUROC: 1.0000 | pxl_AUROC: 0.9778 | F1: 1.000
  padim     seed 0 | img_AUROC: 1.0000 | pxl_AUROC: 0.9705 | F1: 1.000
  ensemble  seed 0 | img_AUROC: 1.0000 | pxl_AUROC: 0.9761 | F1: 1.000

----------------------------------------------------------------------
Running Category: Cable
----------------------------------------------------------------------
[cable] 224 train / 150 test, features (1536, 32, 32), extracted in 8s
  ae        seed 0 | img_AUROC: 0.8632 | pxl_AUROC: 0.9630 | F1: 0.853
  ae_pos    seed 0 | img_AUROC: 0.8596 | pxl_AUROC: 0.9639 | F1: 0.833
  padim     seed 0 | img_AUROC: 0.9505 | pxl_AUROC: 0.9769 | F1: 0.911
  ensemble  seed 0 | img_A

## **ANALYSIS**
- img_auroc : padim performs best
- pxl_auroc : ensemble performs best

**Possible reasons :**

1. ae compress and decompress the features, causing an inherent and unavoidable blurring of features in the output, padim on other hand works without any reconstruction and is hence not prone to noisy pixels arising due to blurring and raising false prediction as in ae

2. padim can sometimes be too sensitive, autoencoders on oother hand give smooth localised errors but might miss faint defects, these 2 cancel each other's noise at pixel level improving pxl_auroc of the ensemble compared to either of the models